In [1]:
pip install pandas numpy faker python-dateutil

Note: you may need to restart the kernel to use updated packages.


In [16]:
from datetime import datetime
from datetime import timedelta
import pandas as pd
import numpy as np
from faker import Faker
import random

In [17]:
fake = Faker("en_IN")
np.random.seed(42)
random.seed(42)

START_DATE = datetime(2023, 1, 1)
END_DATE = datetime(2024, 12, 31)

NUM_PATIENTS = 50000
NUM_DOCTORS = 300
NUM_BRANCHES = 5


In [18]:
date_df = pd.date_range(start=START_DATE, end=END_DATE, freq="D").to_frame(index=False, name="date")
date_df["day"] = date_df["date"].dt.day
date_df["month"] = date_df["date"].dt.month
date_df["year"] = date_df["date"].dt.year
date_df["quarter"] = date_df["date"].dt.quarter
date_df["week"] = date_df["date"].dt.isocalendar().week
date_df["is_weekend"] = date_df["date"].dt.weekday >= 5

In [19]:
branches = []
for i in range(1, NUM_BRANCHES + 1):
    branches.append({
        "branch_id": i,
        "branch_name": f"Hospital Branch {i}",
        "city": fake.city(),
        "total_beds": random.randint(300, 700),
        "icu_beds": random.randint(40, 100),
        "ventilators": random.randint(30, 80)
    })

branch_df = pd.DataFrame(branches)

In [20]:
department_df = pd.DataFrame({
    "department_id": [1,2,3,4,5,6],
    "department_name": [
        "Cardiology",
        "Oncology",
        "Orthopedics",
        "Pediatrics",
        "Emergency",
        "General Medicine"
    ]
})

In [21]:
doctor_rows = []
for i in range(1, NUM_DOCTORS + 1):
    doctor_rows.append({
        "doctor_id": i,
        "doctor_name": fake.name(),
        "department_id": random.choice(department_df.department_id),
        "branch_id": random.randint(1, NUM_BRANCHES),
        "employment_type": random.choice(["Full-time", "Visiting"])
    })

doctor_df = pd.DataFrame(doctor_rows)

In [34]:
patient_rows = []
for i in range(1, NUM_PATIENTS + 1):
    age = random.randint(0, 90)
    patient_rows.append({
        "patient_id": i,
        "age": age,
        "age_group": (
            "0-14" if age < 15 else
            "15-30" if age <= 30 else
            "31-45" if age <= 45 else
            "46-60" if age <= 60 else
            "60+"
        ),
        "gender": random.choice(["Male", "Female"]),
        "insurance_type": random.choice(["Cash", "Govt", "Private"])
    })

patient_df = pd.DataFrame(patient_rows)

In [23]:
admissions = []
admission_id = 1

for _, patient in patient_df.sample(65000, replace=True).iterrows():
    dept = department_df.sample(1).iloc[0]
    doctor = doctor_df[doctor_df.department_id == dept.department_id].sample(1).iloc[0]

    admit_date = fake.date_time_between(start_date=START_DATE, end_date=END_DATE)

    # LOS rules by department
    if dept.department_name == "Oncology":
        los_days = random.randint(7, 20)
    elif dept.department_name == "Emergency":
        los_days = random.randint(1, 3)
    else:
        los_days = random.randint(2, 10)

    discharge_date = admit_date + timedelta(days=los_days)

    admissions.append({
        "admission_id": admission_id,
        "patient_id": patient.patient_id,
        "branch_id": doctor.branch_id,
        "department_id": dept.department_id,
        "doctor_id": doctor.doctor_id,
        "admission_datetime": admit_date,
        "discharge_datetime": discharge_date,
        "admission_type": random.choices(
            ["Emergency", "Scheduled"], weights=[0.4, 0.6]
        )[0],
        "diagnosis_category": random.choice([
            "Cardiac", "Cancer", "Trauma", "Infection", "Respiratory"
        ]),
        "outcome": random.choices(
            ["Recovered", "Improved", "Transferred", "Deceased"],
            weights=[70, 20, 7, 3]
        )[0]
    })

    admission_id += 1

admission_df = pd.DataFrame(admissions)

In [24]:
bed_rows = []

for _, row in branch_df.iterrows():
    for _, dept in department_df.iterrows():
        for _, d in date_df.iterrows():
            occupied = random.randint(
                int(row.total_beds * 0.5),
                int(row.total_beds * 0.95)
            )
            bed_rows.append({
                "date": d.date,
                "branch_id": row.branch_id,
                "department_id": dept.department_id,
                "total_beds": row.total_beds,
                "occupied_beds": occupied,
                "icu_beds": row.icu_beds,
                "icu_occupied": random.randint(
                    int(row.icu_beds * 0.4),
                    int(row.icu_beds * 0.95)
                )
            })

bed_df = pd.DataFrame(bed_rows)


In [25]:
schedule_rows = []

for _, doctor in doctor_df.iterrows():
    for _, d in date_df.sample(300).iterrows():
        booked = random.randint(4, 10)
        schedule_rows.append({
            "doctor_id": doctor.doctor_id,
            "date": d.date,
            "available_hours": 8,
            "booked_hours": booked
        })

schedule_df = pd.DataFrame(schedule_rows)


In [31]:
procedure_rows = []
procedure_id = 1

for _, adm in admission_df.sample(40000).iterrows():
    procedure_rows.append({
        "procedure_id": procedure_id,
        "admission_id": adm.admission_id,
        "department_id": adm.department_id,
        "procedure_type": random.choice(["Surgery", "Scan", "Therapy"]),
        "procedure_datetime": adm.admission_datetime + timedelta(hours=random.randint(1,48)),
        "procedure_cost": random.randint(5000, 150000)
    })
    procedure_id += 1

procedure_df = pd.DataFrame(procedure_rows)

procedure_df


,procedure_id,admission_id,department_id,procedure_type,procedure_datetime,procedure_cost
0,1,46714,3,Therapy,2024-11-30 08:08:15,137739
1,2,10833,2,Surgery,2024-02-13 07:33:33,118954
2,3,3082,4,Surgery,2024-12-18 12:30:41,40229
3,4,46014,4,Scan,2024-07-03 19:48:29,22096
4,5,28780,1,Therapy,2023-04-15 06:54:09,68870
...,...,...,...,...,...,...
39995,39996,64175,3,Surgery,2023-08-05 07:21:17,43047
39996,39997,3996,5,Scan,2023-08-28 20:03:34,39750
39997,39998,14735,2,Surgery,2024-05-21 02:31:23,45610
39998,39999,2973,4,Therapy,2023-12-19 07:48:28,100456


In [29]:
billing_rows = []

for _, adm in admission_df.iterrows():
    total = random.randint(20000, 300000)
    insurance = random.randint(int(total * 0.4), int(total * 0.8))

    billing_rows.append({
        "admission_id": adm.admission_id,
        "total_cost": total,
        "room_charges": int(total * 0.4),
        "procedure_charges": int(total * 0.35),
        "medicine_charges": int(total * 0.25),
        "insurance_covered": insurance,
        "patient_paid": total - insurance
    })

billing_df = pd.DataFrame(billing_rows)

billing_df


,admission_id,total_cost,room_charges,procedure_charges,medicine_charges,insurance_covered,patient_paid
0,1,141588,56635,49555,35397,80258,61330
1,2,165328,66131,57864,41332,115146,50182
2,3,281950,112780,98682,70487,207255,74695
3,4,183587,73434,64255,45896,93983,89604
4,5,207221,82888,72527,51805,121079,86142
...,...,...,...,...,...,...,...
64995,64996,125034,50013,43761,31258,50902,74132
64996,64997,169695,67878,59393,42423,121341,48354
64997,64998,64503,25801,22576,16125,36738,27765
64998,64999,48857,19542,17099,12214,34432,14425


In [32]:
branch_df.to_csv("dim_branch.csv", index=False)
department_df.to_csv("dim_department.csv", index=False)
doctor_df.to_csv("dim_doctor.csv", index=False)
patient_df.to_csv("dim_patient.csv", index=False)
date_df.to_csv("dim_date.csv", index=False)

admission_df.to_csv("fact_admissions.csv", index=False)
bed_df.to_csv("fact_bed_occupancy.csv", index=False)
schedule_df.to_csv("fact_doctor_schedule.csv", index=False)
procedure_df.to_csv("fact_procedures.csv", index=False)
billing_df.to_csv("fact_billing.csv", index=False)